# 21 - Sensitivity analysis 3: forest plots (primary vs competing-event models)

Plots minor-bleeding hazard ratios from the primary analysis (notebooks 04-07) against the competing-event models (notebooks 17-20).

**Manuscript items:** Fig S11a-d

**Inputs:** `RESULTS_DIR/hr_*_minor_summary.xlsx`; `SENS_COMPETING_DIR/hr_*_minor_summary.xlsx`

**Outputs:** `SUPP_FIGURES_DIR/forest_plot_minor_drug_name.png`, `forest_plot_minor_drug_class.png`, `forest_plot_pk_pd.png`, `forest_plot_num_drugs.png`

*Derived from the original analysis notebook `sensitivity_analysis/competing_risk/combine.ipynb`. Outputs have been cleared; no patient-level data are included.*

In [ ]:
# Project file locations are defined once in paths.py (same folder as this notebook).
# Edit DATA_DIR / OUTPUT_DIR there, or set the DDI_DATA_DIR / DDI_OUTPUT_DIR environment variables.
import os
from paths import *

In [ ]:
# Cap BLAS/OpenMP threads BEFORE numpy is imported. Without this, each of the
# worker processes below spawns its own thread pool and the machine thrashes.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import time
import matplotlib.pyplot as plt
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp
from multiprocessing import Pool
from lifelines import CoxTimeVaryingFitter
import zipfile


In [ ]:
COLS = ["drug_class", "Patient Counts", "events/person_years", "HR", "CI Lower", "CI Upper"]

def forest_plot_drug_name(df_a, df_b, label_a, label_b, out_path):
    a = df_a[COLS].copy(); a["model"] = label_a
    b = df_b[COLS].copy(); b["model"] = label_b
    # keep the drug order of df_a, then add any drug only present in df_b
    drugs = list(a["drug_class"]) + [d for d in b["drug_class"] if d not in set(a["drug_class"])]

    rows = []  # (y, drug_label, model, record-or-None)
    y = 0
    for d in drugs:
        rows.append((y, d, None, None)); y += 1            # drug header row
        for lab, df in ((label_a, a), (label_b, b)):
            r = df[df["drug_class"] == d]
            rows.append((y, None, lab, None if r.empty else r.iloc[0])); y += 1
        y += 0.4                                            # gap between drugs
    n = y

    fig = plt.figure(figsize=(13, 0.26 * n + 1.6))
    ax_t = fig.add_axes([0.01, 0.05, 0.58, 0.88])
    ax_f = fig.add_axes([0.61, 0.05, 0.37, 0.88], sharey=ax_t)
    for ax in (ax_t, ax_f):
        ax.set_ylim(n, -1.4)
    ax_t.set_xlim(0, 1); ax_t.axis("off")

    xcol = {"drug": 0.0, "model": 0.03, "pts": 0.42, "ev": 0.58, "hr": 0.80}
    hdr = [("drug", "Drug name", "left"), ("pts", "Patients", "right"),
           ("ev", "Events/PY", "right"), ("hr", "Adjusted HR (95% CI)", "right")]
    xpos = {"pts": 0.52, "ev": 0.70, "hr": 1.0}
    for k, txt, ha in hdr:
        ax_t.text(xpos.get(k, 0.0), -1, txt, ha=ha, va="center", fontweight="bold", fontsize=10)
    ax_t.axhline(-0.5, color="black", lw=0.8)

    colors = {label_a: "#1f77b4", label_b: "#d62728"}
    for y, drug, model, r in rows:
        if drug is not None:
            ax_t.text(0.0, y, drug, fontweight="bold", va="center", fontsize=9)
            continue
        ax_t.text(0.03, y, model, va="center", fontsize=8.5, color=colors[model])
        if r is None:
            ax_t.text(1.0, y, "Not estimated", ha="right", va="center", fontsize=8.5, style="italic", color="grey")
            continue
        ax_t.text(0.52, y, f"{int(r['Patient Counts']):,}", ha="right", va="center", fontsize=8.5)
        ax_t.text(0.70, y, str(r["events/person_years"]), ha="right", va="center", fontsize=8.5)
        ax_t.text(1.0, y, f"{r['HR']:.2f} ({r['CI Lower']:.2f}–{r['CI Upper']:.2f})",
                  ha="right", va="center", fontsize=8.5)
        ax_f.errorbar(r["HR"], y, xerr=[[r["HR"] - r["CI Lower"]], [r["CI Upper"] - r["HR"]]],
                      fmt="s", color=colors[model], ms=4.5, capsize=2.5, lw=1.2)

    ax_f.axvline(1, color="black", ls="--", lw=0.8)
    ax_f.set_xscale("log")
    ticks = [0.7, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5]
    ax_f.set_xticks(ticks); ax_f.set_xticklabels([f"{t:g}" for t in ticks])
    ax_f.minorticks_off()
    ax_f.set_xlim(0.7, 2.6)
    ax_f.set_xlabel("Adjusted hazard ratio (95% CI, log scale)")
    ax_f.tick_params(axis="y", left=False, labelleft=False)
    for s in ("top", "right", "left"):
        ax_f.spines[s].set_visible(False)
    ax_f.text(0.95, -1, "← Lower risk", ha="right", va="center", fontsize=8, color="grey")
    ax_f.text(1.05, -1, "Higher risk →", ha="left", va="center", fontsize=8, color="grey")
    handles = [plt.Line2D([], [], marker="s", ls="", color=c, label=l) for l, c in colors.items()]
    ax_f.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.045), ncol=2, fontsize=8, frameon=False)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    return fig


### Drug Name

In [ ]:
hr_drug_name_no_com = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_name_minor_summary.xlsx"))
hr_drug_name_com = pd.read_excel(os.path.join(SENS_COMPETING_DIR, "hr_drug_name_minor_summary.xlsx"))

In [ ]:
if __name__ == "__main__":
    forest_plot_drug_name(hr_drug_name_no_com, hr_drug_name_com,
                      "Standard Cox model", "Competing-risk model (serious bleeding)",
                      os.path.join(SUPP_FIGURES_DIR, "forest_plot_minor_drug_name.png"))

### Drug Class

In [ ]:
COLS = ["drug_class", "Patient Counts", "events/person_years", "HR", "CI Lower", "CI Upper"]

def forest_plot_drug_class(df_a, df_b, label_a, label_b, out_path):
    a = df_a[COLS].copy(); a["model"] = label_a
    b = df_b[COLS].copy(); b["model"] = label_b
    # keep the drug order of df_a, then add any drug only present in df_b
    drugs = list(a["drug_class"]) + [d for d in b["drug_class"] if d not in set(a["drug_class"])]

    rows = []  # (y, drug_label, model, record-or-None)
    y = 0
    for d in drugs:
        rows.append((y, d, None, None)); y += 1            # drug header row
        for lab, df in ((label_a, a), (label_b, b)):
            r = df[df["drug_class"] == d]
            rows.append((y, None, lab, None if r.empty else r.iloc[0])); y += 1
        y += 0.4                                            # gap between drugs
    n = y

    fig = plt.figure(figsize=(13, 0.26 * n + 1.6))
    ax_t = fig.add_axes([0.01, 0.05, 0.58, 0.88])
    ax_f = fig.add_axes([0.61, 0.05, 0.37, 0.88], sharey=ax_t)
    for ax in (ax_t, ax_f):
        ax.set_ylim(n, -1.4)
    ax_t.set_xlim(0, 1); ax_t.axis("off")

    xcol = {"drug": 0.0, "model": 0.03, "pts": 0.42, "ev": 0.58, "hr": 0.80}
    hdr = [("drug", "Drug class", "left"), ("pts", "Patients", "right"),
           ("ev", "Events/PY", "right"), ("hr", "Adjusted HR (95% CI)", "right")]
    xpos = {"pts": 0.52, "ev": 0.70, "hr": 1.0}
    for k, txt, ha in hdr:
        ax_t.text(xpos.get(k, 0.0), -1, txt, ha=ha, va="center", fontweight="bold", fontsize=10)
    ax_t.axhline(-0.5, color="black", lw=0.8)

    colors = {label_a: "#1f77b4", label_b: "#d62728"}
    for y, drug, model, r in rows:
        if drug is not None:
            ax_t.text(0.0, y, drug, fontweight="bold", va="center", fontsize=9)
            continue
        ax_t.text(0.03, y, model, va="center", fontsize=8.5, color=colors[model])
        if r is None:
            ax_t.text(1.0, y, "Not estimated", ha="right", va="center", fontsize=8.5, style="italic", color="grey")
            continue
        ax_t.text(0.52, y, f"{int(r['Patient Counts']):,}", ha="right", va="center", fontsize=8.5)
        ax_t.text(0.70, y, str(r["events/person_years"]), ha="right", va="center", fontsize=8.5)
        ax_t.text(1.0, y, f"{r['HR']:.2f} ({r['CI Lower']:.2f}–{r['CI Upper']:.2f})",
                  ha="right", va="center", fontsize=8.5)
        ax_f.errorbar(r["HR"], y, xerr=[[r["HR"] - r["CI Lower"]], [r["CI Upper"] - r["HR"]]],
                      fmt="s", color=colors[model], ms=4.5, capsize=2.5, lw=1.2)

    ax_f.axvline(1, color="black", ls="--", lw=0.8)
    ax_f.set_xscale("log")
    ticks = [0.7, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5]
    ax_f.set_xticks(ticks); ax_f.set_xticklabels([f"{t:g}" for t in ticks])
    ax_f.minorticks_off()
    ax_f.set_xlim(0.7, 2.6)
    ax_f.set_xlabel("Adjusted hazard ratio (95% CI, log scale)", fontsize=8)
    ax_f.tick_params(axis="y", left=False, labelleft=False)
    for s in ("top", "right", "left"):
        ax_f.spines[s].set_visible(False)
    ax_f.text(0.95, -1, "← Lower risk", ha="right", va="center", fontsize=8, color="grey")
    ax_f.text(1.05, -1, "Higher risk →", ha="left", va="center", fontsize=8, color="grey")
    handles = [plt.Line2D([], [], marker="s", ls="", color=c, label=l) for l, c in colors.items()]
    ax_f.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.045), ncol=2, fontsize=8, frameon=False)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    return fig


In [ ]:
hr_drug_class_no_com = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_class_minor_summary.xlsx"))
hr_drug_class_com = pd.read_excel(os.path.join(SENS_COMPETING_DIR, "hr_drug_class_minor_summary.xlsx"))

In [ ]:
if __name__ == "__main__":
    forest_plot_drug_class(hr_drug_class_no_com, hr_drug_class_com,
                      "Standard Cox model", "Competing-risk model (serious bleeding)",
                      os.path.join(SUPP_FIGURES_DIR, "forest_plot_minor_drug_class.png"))

### Number of concomitant drugs

In [ ]:
COLS = ["drug_class", "Patient Counts", "events/person_years", "HR", "CI Lower", "CI Upper"]

def forest_plot_num_drugs(df_a, df_b, label_a, label_b, out_path,
                      title="Minor outcome: Adjusted hazard ratios"):
    a = df_a[COLS].copy(); a["model"] = label_a
    b = df_b[COLS].copy(); b["model"] = label_b
    # keep the drug order of df_a, then add any drug only present in df_b
    drugs = list(a["drug_class"]) + [d for d in b["drug_class"] if d not in set(a["drug_class"])]

    rows = []  # (y, drug_label, model, record-or-None)
    y = 0
    for d in drugs:
        rows.append((y, d, None, None)); y += 1            # drug header row
        for lab, df in ((label_a, a), (label_b, b)):
            r = df[df["drug_class"] == d]
            rows.append((y, None, lab, None if r.empty else r.iloc[0])); y += 1
        y += 0.4                                            # gap between drugs
    n = y

    fig = plt.figure(figsize=(13, 0.26 * n + 1.6))
    ax_t = fig.add_axes([0.01, 0.05, 0.58, 0.88])
    ax_f = fig.add_axes([0.61, 0.05, 0.37, 0.88], sharey=ax_t)
    for ax in (ax_t, ax_f):
        ax.set_ylim(n, -1.4)
    ax_t.set_xlim(0, 1); ax_t.axis("off")

    xcol = {"drug": 0.0, "model": 0.03, "pts": 0.42, "ev": 0.58, "hr": 0.80}
    hdr = [("drug", "Number of concomitant drugs", "left"), ("pts", "Patients", "right"),
           ("ev", "Events/PY", "right"), ("hr", "Adjusted HR (95% CI)", "right")]
    xpos = {"pts": 0.52, "ev": 0.70, "hr": 1.0}
    for k, txt, ha in hdr:
        ax_t.text(xpos.get(k, 0.0), -1, txt, ha=ha, va="center", fontweight="bold", fontsize=10)
    ax_t.axhline(-0.5, color="black", lw=0.8)

    colors = {label_a: "#1f77b4", label_b: "#d62728"}
    for y, drug, model, r in rows:
        if drug is not None:
            ax_t.text(0.0, y, drug, fontweight="bold", va="center", fontsize=10)
            continue
        ax_t.text(0.03, y, model, va="center", fontsize=8.5, color=colors[model])
        if r is None:
            ax_t.text(1.0, y, "Not estimated", ha="right", va="center", fontsize=8.5, style="italic", color="grey")
            continue
        ax_t.text(0.52, y, f"{int(r['Patient Counts']):,}", ha="right", va="center", fontsize=8.5)
        ax_t.text(0.70, y, str(r["events/person_years"]), ha="right", va="center", fontsize=8.5)
        ax_t.text(1.0, y, f"{r['HR']:.2f} ({r['CI Lower']:.2f}–{r['CI Upper']:.2f})",
                  ha="right", va="center", fontsize=8.5)
        ax_f.errorbar(r["HR"], y, xerr=[[r["HR"] - r["CI Lower"]], [r["CI Upper"] - r["HR"]]],
                      fmt="s", color=colors[model], ms=4.5, capsize=2.5, lw=1.2)

    ax_f.axvline(1, color="black", ls="--", lw=0.8)
    ax_f.set_xscale("log")
    ticks = [0.7, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5]
    ax_f.set_xticks(ticks); ax_f.set_xticklabels([f"{t:g}" for t in ticks])
    ax_f.minorticks_off()
    ax_f.set_xlim(0.7, 2.6)
    ax_f.set_xlabel("Adjusted hazard ratio (95% CI, log scale)", fontsize=8)
    ax_f.tick_params(axis="y", left=False, labelleft=False)
    for s in ("top", "right", "left"):
        ax_f.spines[s].set_visible(False)
    ax_f.text(0.95, -1, "← Lower risk", ha="right", va="center", fontsize=8, color="grey")
    ax_f.text(1.05, -1, "Higher risk →", ha="left", va="center", fontsize=8, color="grey")
    handles = [plt.Line2D([], [], marker="s", ls="", color=c, label=l) for l, c in colors.items()]
    ax_f.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=8, frameon=False)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    return fig


In [ ]:
hr_drug_num_drugs_no_com = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_num_minor_summary.xlsx"))
hr_drug_num_drugs_com = pd.read_excel(os.path.join(SENS_COMPETING_DIR, "hr_drug_num_minor_summary.xlsx"))


In [ ]:
hr_drug_num_drugs_no_com.rename(columns={'patient_counts': 'Patient Counts'}, inplace=True)
hr_drug_num_drugs_no_com["drug_class"] = hr_drug_num_drugs_no_com["drug_class"].replace({
    "1_Drug": "1 Drug",
    "2_Drugs": "2 Drugs",
    "3_or_more_Drugs": "≥3 Drugs"
})
hr_drug_num_drugs_com["drug_class"] = hr_drug_num_drugs_com["drug_class"].replace({
    "1_Drug": "1 Drug",
    "2_Drugs": "2 Drugs",
    "3_or_more_Drugs": "≥3 Drugs"
})

In [ ]:
if __name__ == "__main__":
    forest_plot_num_drugs(hr_drug_num_drugs_no_com, hr_drug_num_drugs_com,
                      "Standard Cox model", "Competing-risk model (serious bleeding)",
                      os.path.join(SUPP_FIGURES_DIR, "forest_plot_num_drugs.png"))

### PK/PD

In [ ]:
COLS = ["drug_class", "Patient Counts", "events/person_years", "HR", "CI Lower", "CI Upper"]

def forest_plot_pk_pd(df_a, df_b, label_a, label_b, out_path,
                      title="Minor outcome: Adjusted hazard ratios"):
    a = df_a[COLS].copy(); a["model"] = label_a
    b = df_b[COLS].copy(); b["model"] = label_b
    # keep the drug order of df_a, then add any drug only present in df_b
    drugs = list(a["drug_class"]) + [d for d in b["drug_class"] if d not in set(a["drug_class"])]

    rows = []  # (y, drug_label, model, record-or-None)
    y = 0
    for d in drugs:
        rows.append((y, d, None, None)); y += 1            # drug header row
        for lab, df in ((label_a, a), (label_b, b)):
            r = df[df["drug_class"] == d]
            rows.append((y, None, lab, None if r.empty else r.iloc[0])); y += 1
        y += 0.4                                            # gap between drugs
    n = y

    fig = plt.figure(figsize=(13, 0.26 * n + 1.6))
    ax_t = fig.add_axes([0.01, 0.05, 0.58, 0.88])
    ax_f = fig.add_axes([0.61, 0.05, 0.37, 0.88], sharey=ax_t)
    for ax in (ax_t, ax_f):
        ax.set_ylim(n, -1.4)
    ax_t.set_xlim(0, 1); ax_t.axis("off")

    xcol = {"drug": 0.0, "model": 0.03, "pts": 0.42, "ev": 0.58, "hr": 0.80}
    hdr = [("drug", "Interaction mechanism", "left"), ("pts", "Patients", "right"),
           ("ev", "Events/PY", "right"), ("hr", "Adjusted HR (95% CI)", "right")]
    xpos = {"pts": 0.52, "ev": 0.70, "hr": 1.0}
    for k, txt, ha in hdr:
        ax_t.text(xpos.get(k, 0.0), -1, txt, ha=ha, va="center", fontweight="bold", fontsize=10)
    ax_t.axhline(-0.5, color="black", lw=0.8)

    colors = {label_a: "#1f77b4", label_b: "#d62728"}
    for y, drug, model, r in rows:
        if drug is not None:
            ax_t.text(0.0, y, drug, fontweight="bold", va="center", fontsize=10)
            continue
        ax_t.text(0.03, y, model, va="center", fontsize=8.5, color=colors[model])
        if r is None:
            ax_t.text(1.0, y, "Not estimated", ha="right", va="center", fontsize=8.5, style="italic", color="grey")
            continue
        ax_t.text(0.52, y, f"{int(r['Patient Counts']):,}", ha="right", va="center", fontsize=8.5)
        ax_t.text(0.70, y, str(r["events/person_years"]), ha="right", va="center", fontsize=8.5)
        ax_t.text(1.0, y, f"{r['HR']:.2f} ({r['CI Lower']:.2f}–{r['CI Upper']:.2f})",
                  ha="right", va="center", fontsize=8.5)
        ax_f.errorbar(r["HR"], y, xerr=[[r["HR"] - r["CI Lower"]], [r["CI Upper"] - r["HR"]]],
                      fmt="s", color=colors[model], ms=4.5, capsize=2.5, lw=1.2)

    ax_f.axvline(1, color="black", ls="--", lw=0.8)
    ax_f.set_xscale("log")
    ticks = [0.7, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5]
    ax_f.set_xticks(ticks); ax_f.set_xticklabels([f"{t:g}" for t in ticks])
    ax_f.minorticks_off()
    ax_f.set_xlim(0.7, 2.6)
    ax_f.set_xlabel("Adjusted hazard ratio (95% CI, log scale)", fontsize=8)
    ax_f.tick_params(axis="y", left=False, labelleft=False)
    for s in ("top", "right", "left"):
        ax_f.spines[s].set_visible(False)
    ax_f.text(0.95, -1, "← Lower risk", ha="right", va="center", fontsize=8, color="grey")
    ax_f.text(1.05, -1, "Higher risk →", ha="left", va="center", fontsize=8, color="grey")
    handles = [plt.Line2D([], [], marker="s", ls="", color=c, label=l) for l, c in colors.items()]
    ax_f.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=8, frameon=False)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    return fig


In [ ]:
hr_drug_pk_pd_no_com = pd.read_excel(os.path.join(RESULTS_DIR, "hr_drug_pk_pd_minor_summary.xlsx"))
hr_drug_pd_pd_com = pd.read_excel(os.path.join(SENS_COMPETING_DIR, "hr_drug_pk_pd_minor_summary.xlsx"))
hr_drug_pk_pd_no_com.rename(columns={'patient_counts': 'Patient Counts'}, inplace=True)
hr_drug_pk_pd_no_com["drug_class"] = hr_drug_pk_pd_no_com["drug_class"].replace({"p_d_drug": "PD drug", "p_k_drug": "PK drug"})
hr_drug_pd_pd_com["drug_class"] = hr_drug_pd_pd_com["drug_class"].replace({"p_d_drug": "PD drug", "p_k_drug": "PK drug"})

In [ ]:
if __name__ == "__main__":
    forest_plot_pk_pd(hr_drug_pk_pd_no_com, hr_drug_pd_pd_com,
                      "Standard Cox model", "Competing-risk model (serious bleeding)",
                      os.path.join(SUPP_FIGURES_DIR, "forest_plot_pk_pd.png"))